# LangChain Document Loaders

Document loaders turn files and other sources into LangChain `Document` objects. A document has `page_content` and `metadata`. This notebook introduces loaders for text, Markdown, CSV, JSON, directories, PDF, Word, and web pages.

Run the setup cell first. PDF, Word, and web examples need their listed extra packages and (for the web example) an internet connection.

## 1. Install dependencies

The project pins LangChain in `requirements.txt`. Install the community loaders and format-specific extras in the active environment:

In [8]:
%pip install langchain-community
%pip install pypdf docx2txt beautifulsoup4
%pip install jq

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import json

from langchain_community.document_loaders import (
    CSVLoader, DirectoryLoader, Docx2txtLoader, PyPDFLoader,
    TextLoader, WebBaseLoader, JSONLoader,
)
from langchain_core.documents import Document

DATA_DIR = Path('loader_samples')
DATA_DIR.mkdir(exist_ok=True)
print('Ready. Sample data will be written to:', DATA_DIR.resolve())

Ready. Sample data will be written to: /Users/kshitijberi/Documents/GitHub/learning_rag/document_loaders/loader_samples


## 2. Text files with `TextLoader`

Use this for a single plain text file. Set the encoding explicitly when you know it.

In [10]:
text_path = DATA_DIR / 'notes.txt'
text_path.write_text("""RAG combines retrieval with generation.
Documents provide context to a language model.""", encoding='utf-8')

text_docs = TextLoader(str(text_path), encoding='utf-8').load()
print(text_docs[0].page_content)
print('Metadata:', text_docs[0].metadata)

RAG combines retrieval with generation.
Documents provide context to a language model.
Metadata: {'source': 'loader_samples/notes.txt'}


## 3. CSV files with `CSVLoader`

By default, each CSV row becomes one document. `metadata_columns` keeps selected fields available as metadata.

In [11]:
csv_path = DATA_DIR / 'courses.csv'
csv_path.write_text("""course,topic
RAG Basics,retrieval augmented generation
Embeddings,vector representations
""", encoding='utf-8')

csv_docs = CSVLoader(str(csv_path), metadata_columns=['course']).load()
print(csv_docs[0].page_content)
print('Metadata:', csv_docs[0].metadata)

topic: retrieval augmented generation
Metadata: {'source': 'loader_samples/courses.csv', 'row': 0, 'course': 'RAG Basics'}


## 4. JSON files with `JSONLoader`

`jq_schema` selects the records to load. Install `jq` (`%pip install jq`) if this example reports that the `jq` package is missing.

In [12]:
json_path = DATA_DIR / 'articles.json'
json_path.write_text(json.dumps({'articles': [
    {'title': 'Chunking', 'body': 'Split long text into useful passages.'},
    {'title': 'Retrieval', 'body': 'Find relevant passages for a query.'},
]}, indent=2), encoding='utf-8')

json_docs = JSONLoader(
    file_path=str(json_path),
    jq_schema='.articles[]',
    content_key='body',
    metadata_func=lambda record, metadata: {**metadata, 'title': record.get('title')},
).load()
print(json_docs[0].page_content)
print('Metadata:', json_docs[0].metadata)

Split long text into useful passages.
Metadata: {'source': '/Users/kshitijberi/Documents/GitHub/learning_rag/document_loaders/loader_samples/articles.json', 'seq_num': 1, 'title': 'Chunking'}


## 5. Load many files with `DirectoryLoader`

A directory loader finds files using a glob and applies a loader to each matching file. Here it reuses `TextLoader` for all `.txt` files.

In [13]:
(DATA_DIR / 'second.txt').write_text('A second document for the collection.', encoding='utf-8')
directory_docs = DirectoryLoader(
    str(DATA_DIR), glob='*.txt', loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
).load()
print(f'Loaded {len(directory_docs)} text documents')
print(directory_docs[0].metadata)

Loaded 2 text documents
{'source': 'loader_samples/notes.txt'}


## 6. Markdown, PDF, and Word documents

Markdown can be read as plain text with `TextLoader`. `PyPDFLoader` typically creates one document per page. `Docx2txtLoader` extracts text from a Word `.docx` file. Install `pypdf` and `docx2txt` as shown above. Provide your own file paths to run the PDF and Word examples.

In [14]:
md_path = DATA_DIR / 'guide.md'
md_path.write_text('# Loader notes\n\nMarkdown is also readable text.', encoding='utf-8')
markdown_docs = TextLoader(str(md_path), encoding='utf-8').load()
print(markdown_docs[0].page_content)

# Change these paths to files that exist on your machine, then run the cells.
# pdf_docs = PyPDFLoader('path/to/your/document.pdf').load()
# word_docs = Docx2txtLoader('path/to/your/document.docx').load()

# Loader notes

Markdown is also readable text.


## 7. Web pages with `WebBaseLoader`

This fetches a page and extracts its text. It needs network access; replace the URL with a page you are allowed to access.

In [15]:
web_docs = WebBaseLoader('https://en.wikipedia.org/wiki/Retrieval-augmented_generation').load()
print(web_docs[0].page_content[100:500])
print('Metadata:', web_docs[0].metadata)

in menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create ac
Metadata: {'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en'}


## 8. A loader can return multiple documents

Inspect both content and metadata before splitting and indexing. Loaders do not all use the same document boundaries: a CSV loader often emits one document per row, while a PDF loader often emits one per page.

In [16]:
for doc in text_docs + csv_docs + json_docs:
    print('---')
    print(doc.page_content[:160])
    print(doc.metadata)

---
RAG combines retrieval with generation.
Documents provide context to a language model.
{'source': 'loader_samples/notes.txt'}
---
topic: retrieval augmented generation
{'source': 'loader_samples/courses.csv', 'row': 0, 'course': 'RAG Basics'}
---
topic: vector representations
{'source': 'loader_samples/courses.csv', 'row': 1, 'course': 'Embeddings'}
---
Split long text into useful passages.
{'source': '/Users/kshitijberi/Documents/GitHub/learning_rag/document_loaders/loader_samples/articles.json', 'seq_num': 1, 'title': 'Chunking'}
---
Find relevant passages for a query.
{'source': '/Users/kshitijberi/Documents/GitHub/learning_rag/document_loaders/loader_samples/articles.json', 'seq_num': 2, 'title': 'Retrieval'}
